[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coopster-seclusion/sar-optical-pipeline/blob/main/notebooks/03_hyp3_retrieval.ipynb)

# Notebook 03: HyP3 RTC-S1 Retrieval

Searches ASF DAAC for Sentinel-1 IW GRD scenes over the study AOI, submits on-demand RTC jobs
via HyP3 (γ⁰, 10 m, COP-DEM, no speckle filter), then downloads, composites (temporal median,
power domain), converts to dB, clips to the AOI, and writes to Google Drive.

All site parameters come from `config.yaml` — no hardcoded values in this notebook.

**Pilot run:** `PILOT_YEARS` in the setup cell limits processing to a subset of epochs.
Set it to `None` to process all epochs once the workflow is validated.

**Requires Colab Secrets:** `EARTHDATA_USERNAME`, `EARTHDATA_PASSWORD`

**Output:** `data/hyp3/hyp3_{year}_{pol}.tif` — Float32 dB rasters, clipped to AOI, nodata=NaN

---
### Known uncertainties — read before running

- **Polarization:** Orbit- and region-dependent. Arctic sites often yield HH+HV; mid-latitude
  sites typically yield VV+VH. Run the **Inspect results** cell before submitting and update
  `config.yaml` `opera.polarizations` if scenes differ from expected.
- **2016 scene availability:** S1B launched April 2016. Aug–Sep 2016 coverage may be limited
  to 1–2 scenes from S1A only.
- **HyP3 processing time:** ~10–30 min per scene. The monitor cell blocks until all jobs
  complete. If the session disconnects, run the **Recover existing jobs** cell to reload
  state without resubmitting.
- **hyp3_sdk API:** Parameter names (`dem_name`, `scale`, `resolution`) are based on
  hyp3_sdk ≥1.0. If a parameter is rejected, check https://hyp3-api.asf.alaska.edu/ui
  for current accepted values.

## Setup

In [ ]:
# ── Cell 1: run this first, every session ─────────────────────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, subprocess, sys

REPO_URL = 'https://github.com/coopster-seclusion/sar-optical-pipeline.git'
REPO_DIR = '/content/sar-optical-pipeline'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'rioxarray', 'asf_search', 'hyp3_sdk', 'pyogrio'], check=True)

import numpy as np
import rioxarray as rxr
import geopandas as gpd
import matplotlib.pyplot as plt
import asf_search as asf

from pipeline.env import setup, repo_root
from pipeline.validate import validate_config
from pipeline.utils import resolve_crs
import pipeline.hyp3 as hyp3_lib

cfg, DATA, OUT = setup(colab=True)
validate_config(cfg, repo_root())

SITE      = cfg['site_name']
DATA_HYP3 = DATA / 'hyp3'
OUT_FIGS  = OUT  / 'figures'

# ── Epoch filter ───────────────────────────────────────────────────────────────────────
# Set to None to process all epochs in config.yaml.
# List specific years to limit the initial run and validate the workflow first.
PILOT_YEARS  = None   # e.g. {2016, 2025} for just baseline + latest
pilot_epochs = [
    e for e in cfg['epochs']
    if PILOT_YEARS is None or e['year'] in PILOT_YEARS
]

print(f'Site        : {SITE}')
print(f'Data → HyP3 : {DATA_HYP3}')
print(f'Job prefix  : {cfg["hyp3"]["job_prefix"]}')
print(f'Pilot epochs: {[e["year"] for e in pilot_epochs]}')

## Load AOI and resolve CRS

In [ ]:
# ── Cell 2: load AOI and resolve output CRS ─────────────────────────────────────────
ROOT     = repo_root()
aoi_path = ROOT / cfg['aoi']['change_area']
aoi_gdf  = gpd.read_file(aoi_path).to_crs('EPSG:4326')
crs      = resolve_crs(cfg, aoi_path)

print(f'AOI file   : {aoi_path}')
print(f'AOI bounds : {aoi_gdf.total_bounds.round(4)}  (W S E N)')
print(f'Output CRS : {crs}')

## Authenticate Earthdata

Both the HyP3 job client and the ASF scene search use Earthdata Login credentials.
Set `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` in Colab → Secrets before running.

In [ ]:
# ── Cell 3: authenticate Earthdata (HyP3 + ASF scene search) ──────────────────────
username = userdata.get('EARTHDATA_USERNAME')
password = userdata.get('EARTHDATA_PASSWORD')

hyp3    = hyp3_lib.connect(username, password)
session = asf.ASFSession().auth_with_creds(username, password)

print('HyP3 client    : ready')
print('ASF session    : authenticated')

## Search Sentinel-1 GRD scenes

Searches ASF for S1 IW GRD_HD scenes intersecting the AOI for each pilot epoch.
Orbit direction is read from `config.yaml` (`opera.orbit`); the same orbit that covers
the OPERA burst also applies to the HyP3 full-scene GRD search.

**Polarization note:** Check the Inspect cell output before submitting. If scenes report
VV+VH instead of HH+HV, update `config.yaml` `opera.polarizations: [VV, VH]` before
running downstream notebooks.

In [ ]:
# ── Cell 4: search Sentinel-1 IW GRD scenes per epoch ───────────────────────────
orbit = cfg['opera'].get('orbit') or 'descending'

print(f'Orbit: {orbit.upper()}\n')

scene_results = {}
for epoch in pilot_epochs:
    year    = epoch['year']
    results = hyp3_lib.search_s1_scenes(
        aoi_path=aoi_path,
        date_start=epoch['date_start'],
        date_end=epoch['date_end'],
        orbit=orbit,
    )
    scene_results[year] = results
    pols = {r.properties.get('polarization', 'unknown') for r in results} if results else {'none'}
    print(f'  {year}: {len(results):>2} scene(s)   polarization={pols}')

if any(len(r) == 0 for r in scene_results.values()):
    print('\nWARNING: one or more epochs returned 0 scenes.')
    print('If 2016 is empty, S1B had limited coverage before May 2016 (S1A only).')
    print('If other years are empty, try removing flightDirection in pipeline/hyp3.py '
          'or check that the AOI intersects the satellite swath.')

## Inspect results

Verify polarization and path number before submitting. Mixed path numbers are expected
(multiple orbital paths may intersect the AOI) and are fine for compositing.

In [ ]:
# ── Cell 5: inspect scene properties ───────────────────────────────────────────────
print('Detailed scene inspection:\n')
for year, results in scene_results.items():
    print(f'--- {year} ({len(results)} scene(s)) ---')
    if not results:
        print('  No scenes found')
        continue
    for r in results:
        p = r.properties
        print(f"  {p.get('sceneName', p.get('fileID', 'n/a'))}")
        print(f"    acquired:     {p.get('startTime', 'n/a')}")
        print(f"    polarization: {p.get('polarization', 'n/a')}")
        print(f"    flightDir:    {p.get('flightDirection', 'n/a')}")
        print(f"    pathNumber:   {p.get('pathNumber', 'n/a')}")
    print()

## Estimate processing time and submit HyP3 RTC jobs

One job per GRD scene. HyP3 RTC parameters come from `config.yaml` (`hyp3.*`):
- `radiometry: gamma0` — terrain-flattened γ⁰ (matches OPERA)
- `scale: power` — linear power output; composite in power domain then convert to dB
- `resolution: 10` — 10 m output pixel spacing
- `dem: copernicus` — COP-DEM GLO-30 (matches OPERA terrain correction)
- `speckle_filter: false`

Job names are built from `config.yaml` (`hyp3.job_prefix` + year) so they can be
recovered after a session disconnect without resubmitting. Scenes that already have a
SUCCEEDED job in HyP3 are automatically skipped.

In [ ]:
# ── Cell 6: estimate processing time and submit HyP3 RTC jobs ─────────────────────
total_scenes = sum(len(scene_results.get(e['year'], [])) for e in pilot_epochs)
est_min = total_scenes * 10
est_max = total_scenes * 30
print(f'Total scenes across pilot epochs : {total_scenes}')
print(f'Estimated HyP3 processing time   : {est_min}–{est_max} min '
      f'({est_min // 60}–{est_max // 60} h)\n')

if total_scenes > 50:
    print(f'WARNING: {total_scenes} scenes is a large batch (~{est_max // 60}+ hours).')
    print('Consider setting PILOT_YEARS in Cell 1 to process a subset first.\n')

submitted_jobs = {}

for epoch in pilot_epochs:
    year    = epoch['year']
    results = scene_results.get(year, [])

    if not results:
        print(f'{year}: no scenes found — cannot submit')
        submitted_jobs[year] = []
        continue

    pols = cfg['opera'].get('polarizations') or []
    if pols and all((DATA_HYP3 / f'hyp3_{year}_{p}.tif').exists() for p in pols):
        print(f'{year}: output files already exist — skipping submission')
        submitted_jobs[year] = []
        continue

    jobs = hyp3_lib.submit_jobs(
        scenes=results,
        cfg=cfg,
        hyp3_conn=hyp3,
        year=year,
    )
    submitted_jobs[year] = jobs
    n_scenes = len(results)
    print(f'{year}: {len(jobs)} total job(s) queued '
          f'(~{n_scenes * 10}–{n_scenes * 30} min)')

print('\nAll epochs submitted. Run the Monitor cell next.')

## Recover existing jobs (run only after a session disconnect)

If Colab reset after job submission, run this cell to reload job state from HyP3
without resubmitting. Before running:
1. Re-run **Cell 1** (bootstrap) to restore imports and `pilot_epochs`.
2. Re-run **Cell 3** (auth) to restore the `hyp3` client.
3. Run this cell in place of Cell 6.

Skip this cell if Cell 6 ran successfully in the current session.

In [ ]:
# ── Cell 7: recover existing jobs after a session disconnect ────────────────────────
# Run this INSTEAD of Cell 6 after a session reset.
# Job prefix is built from config.yaml (hyp3.job_prefix + year) — no hardcoded strings.

submitted_jobs = {}
for epoch in pilot_epochs:
    year   = epoch['year']
    prefix = f"{cfg['hyp3']['job_prefix']}-{year}"
    try:
        batch = hyp3.find_jobs(name=prefix, job_type='RTC_GAMMA')
        jobs  = list(batch)
        submitted_jobs[year] = jobs
        statuses = {j.status_code for j in jobs}
        print(f'{year}: recovered {len(jobs)} job(s)   statuses={statuses}')
    except Exception as e:
        print(f'{year}: could not recover jobs: {e}')
        submitted_jobs[year] = []

## Monitor job completion

Polls HyP3 every 60 seconds until all jobs reach a terminal state (SUCCEEDED or FAILED).
Timeout is 2 hours per epoch — adjust `timeout` in `pipeline/hyp3.py` for very large batches.

If the session disconnects during monitoring, re-run Cells 1 and 3, run the
**Recover existing jobs** cell, then re-run this cell.

In [ ]:
# ── Cell 8: monitor job completion ─────────────────────────────────────────────────
completed_jobs = {}

for epoch in pilot_epochs:
    year = epoch['year']
    jobs = submitted_jobs.get(year, [])

    if not jobs:
        print(f'{year}: no jobs to watch')
        completed_jobs[year] = []
        continue

    all_done = all(j.status_code in ('SUCCEEDED', 'FAILED') for j in jobs)
    if all_done:
        n_ok  = sum(1 for j in jobs if j.status_code == 'SUCCEEDED')
        n_err = sum(1 for j in jobs if j.status_code == 'FAILED')
        print(f'{year}: already complete — {n_ok} succeeded, {n_err} failed — skipping watch')
        completed_jobs[year] = [j for j in jobs if j.status_code == 'SUCCEEDED']
        continue

    pending = sum(1 for j in jobs if j.status_code not in ('SUCCEEDED', 'FAILED'))
    print(f'{year}: watching {len(jobs)} job(s) ({pending} pending)...')

    succeeded = hyp3_lib.watch_jobs(hyp3, jobs)
    n_failed  = len(jobs) - len(succeeded)
    print(f'  {year}: {len(succeeded)} succeeded, {n_failed} failed')

    completed_jobs[year] = succeeded

print('\nMonitoring complete. Run the Process cell next.')

## Download, composite, dB-convert, clip, save

Per epoch:
1. Downloads all HyP3 ZIP files to a temporary directory
2. Temporal median composite in linear power domain (reduces speckle)
3. Converts to dB: `10 × log10(gamma0_linear)`; zero/negative → NaN
4. Reprojects to output CRS, clips to study AOI
5. Saves Float32 GeoTIFF to `data/hyp3/`; removes temp files

In [ ]:
# ── Cell 9: download, composite, dB-convert, clip, save ───────────────────────────
# polarizations=None lets the module auto-detect from the downloaded TIF filenames.
# Set cfg['opera']['polarizations'] in config.yaml to restrict to specific bands.
pols = cfg['opera'].get('polarizations')   # None = auto-detect

for epoch in pilot_epochs:
    year = epoch['year']
    jobs = completed_jobs.get(year, [])

    if not jobs:
        print(f'  {year}: no succeeded jobs — skipping')
        continue

    print(f'  {year}: processing {len(jobs)} job(s)...')
    outputs = hyp3_lib.download_and_process(
        jobs=jobs,
        aoi_path=aoi_path,
        crs=crs,
        out_dir=DATA,
        year=year,
        polarizations=pols,
    )
    for pol, path in outputs.items():
        arr  = rxr.open_rasterio(path, masked=True).squeeze()
        vmin, vmax = np.nanpercentile(arr.values, [2, 98])
        print(f'    {pol}: shape={arr.shape}  '
              f'2–98pct=[{vmin:.1f}, {vmax:.1f}] dB  → {path.name}')

print('\nDone.')

## Verify baseline epoch

Loads the baseline epoch (from `config.yaml` `change_detection.baseline_year`) and plots
a greyscale preview for each polarization. A clean image with visible land features confirms
correct dB conversion and AOI clipping.

In [ ]:
# ── Cell 10: verify baseline epoch ────────────────────────────────────────────────
baseline = cfg['change_detection']['baseline_year']
pols     = cfg['opera'].get('polarizations') or ['HH', 'HV']
n_pols   = len(pols)

fig, axes = plt.subplots(1, n_pols, figsize=(6 * n_pols, 5))
if n_pols == 1:
    axes = [axes]

any_found = False
for ax, pol in zip(axes, pols):
    path = DATA_HYP3 / f'hyp3_{baseline}_{pol}.tif'
    if not path.exists():
        ax.set_title(f'{baseline} {pol} — NOT FOUND')
        ax.axis('off')
        print(f'Missing: {path}')
        continue

    arr  = rxr.open_rasterio(path, masked=True).squeeze()
    vmin, vmax = np.nanpercentile(arr.values, [2, 98])

    ax.imshow(arr.values, cmap='gray', vmin=vmin, vmax=vmax)
    ax.set_title(f'HyP3 {baseline} {pol} (dB) — {SITE}')
    ax.axis('off')

    print(f'{baseline} {pol}: shape={arr.shape}  CRS={arr.rio.crs}')
    print(f'  range=[{np.nanmin(arr.values):.1f}, {np.nanmax(arr.values):.1f}] dB  '
          f'2–98 pct=[{vmin:.1f}, {vmax:.1f}] dB')
    any_found = True

if any_found:
    plt.tight_layout()
    OUT_FIGS.mkdir(parents=True, exist_ok=True)
    save_path = OUT_FIGS / f'hyp3_{baseline}_preview.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nSaved: {save_path}')
else:
    print('No output files found — run the processing cells above first.')